# 01 · Bases de datos: PostgreSQL + TimescaleDB — PowerShift AI
Este notebook implementa la capa de base de datos del proyecto: crea el esquema relacional de 6 tablas, convierte `telemetria_consumo` en una *hypertable* de TimescaleDB, configura la compresión y la retención, carga los datos reales y ejecuta el **join temporal** entre la telemetría de consumo y las tarifas de CENACE.

**Entradas** (generadas por los notebooks anteriores o descargadas de la fuente):
- `esquema_powershift.sql`: definición de las tablas, la hypertable y las políticas.
- `telemetria_15min.csv`: telemetría limpia de la vivienda `DE_KN_residential4` a 15 minutos, en kW (notebook 00, a partir de OPSD Household Data).
- `programacion.csv` y `costos_diarios.csv`: horario del optimizador para el vehículo eléctrico y costos diarios sin y con orquestador (notebook 04).
- `PMLHorarioMDA.csv`: PML MDA promedio horario por sistema, de CENACE SIM (un solo día).

**Entorno.** El notebook corre en Google Colab. Como Colab no trae un servidor de base de datos, la sección 0 instala PostgreSQL 16 con la extensión TimescaleDB dentro de la propia sesión.

**Qué se ejecuta y qué no.** Todo lo de este notebook se ejecuta de verdad contra la base de datos. El broker MQTT y los medidores e interruptores IoT no existen en el prototipo (no hay hardware) y quedan como diseño en el reporte.

**Supuestos del escenario:**
1. La vivienda de OPSD (Alemania) se asigna al nodo `SIN-PROM`, que representa el PML promedio del Sistema Interconectado Nacional.
2. El PML disponible es de **un solo día**, el 22 de octubre de 2026; la vista `v_perfil_pml` lo usa como perfil horario típico. El cruce se hace por la hora del reloj de pared de la vivienda: hora local + 1 equivale a la hora 1 a 24 de CENACE.

In [1]:
!pip -q install psycopg2-binary

In [5]:
# Celda 2: subir los 5 archivos (se borran cuando Colab se desconecta; hay que volver a subirlos)
from google.colab import files
print("Sube: esquema_powershift.sql, telemetria_15min.csv, programacion.csv, costos_diarios.csv, PMLHorarioMDA.csv")
subidos = files.upload()

Sube: esquema_powershift.sql, telemetria_15min.csv, programacion.csv, costos_diarios.csv, PMLHorarioMDA.csv


Saving costos_diarios.csv to costos_diarios.csv
Saving esquema_powershift.sql to esquema_powershift.sql
Saving PMLHorarioMDA.csv to PMLHorarioMDA.csv
Saving programacion.csv to programacion.csv
Saving telemetria_15min.csv to telemetria_15min.csv


### 0.1 Instalación de PostgreSQL 16 con TimescaleDB
Colab no incluye un servidor de base de datos, por lo que se instala uno dentro de la sesión:
1. Se agregan los repositorios oficiales de PostgreSQL (PGDG) y de Timescale.
2. Se instalan `postgresql-16` y la extensión `timescaledb-2-postgresql-16`.
3. Se precarga la extensión con `shared_preload_libraries = 'timescaledb'`, requisito para que TimescaleDB funcione, y se reinicia el servicio.
4. Se asigna una contraseña al usuario `postgres`. La conexión posterior usa `localhost`, puerto 5432.

Los avisos `W: Skipping acquire of configured file ...` que muestra `apt` corresponden a repositorios propios de Colab y no afectan la instalación.

In [6]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq gnupg wget lsb-release postgresql-common > /dev/null
yes | /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh > /dev/null
echo "deb https://packagecloud.io/timescale/timescaledb/ubuntu/ $(lsb_release -c -s) main" > /etc/apt/sources.list.d/timescaledb.list
wget -q -O - https://packagecloud.io/timescale/timescaledb/gpgkey | gpg --dearmor --yes -o /etc/apt/trusted.gpg.d/timescaledb.gpg
apt-get update -qq
apt-get install -y -qq postgresql-16 timescaledb-2-postgresql-16 > /dev/null
echo "shared_preload_libraries = 'timescaledb'" >> /etc/postgresql/16/main/postgresql.conf
service postgresql restart
sleep 5
su postgres -c "psql -c \"ALTER USER postgres PASSWORD 'powershift';\""
echo "PostgreSQL con TimescaleDB listo (host localhost, puerto 5432, usuario postgres, contraseña powershift)"

 * Restarting PostgreSQL 16 database server
   ...done.
ALTER ROLE
PostgreSQL con TimescaleDB listo (host localhost, puerto 5432, usuario postgres, contraseña powershift)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### 0.2 Librerías y parámetros de conexión
Los parámetros de conexión coinciden con los del servidor recién instalado. `FECHA_PML` es el día al que corresponde el archivo de precios de CENACE.

In [7]:
import os, io, datetime as dt
import numpy as np
import pandas as pd
import psycopg2

DB = dict(host=os.getenv("PGHOST", "localhost"), port=int(os.getenv("PGPORT", "5432")),
          dbname=os.getenv("PGDATABASE", "postgres"), user=os.getenv("PGUSER", "postgres"),
          password=os.getenv("PGPASSWORD", "powershift"))

VIVIENDA = "DE_KN_residential4"
ZONA = "Europe/Berlin"
FECHA_PML = dt.date(2026, 10, 22)   # día de PML que contiene PMLHorarioMDA.csv (descargado el 2-nov-2026)

## 1. Preparación de los datos (solo pandas, aún sin base de datos)

In [8]:
# --- PML de CENACE: HORA 1..24 y una columna por sistema (MXN/MWh)
pml = pd.read_csv("PMLHorarioMDA.csv", skiprows=4, skipinitialspace=True)
pml.columns = [c.strip() for c in pml.columns]
pml = pml.dropna(subset=["HORA"])
pml["HORA"] = pml["HORA"].astype(int)
tarifa = pml.melt(id_vars="HORA", value_vars=["SIN", "BCA", "BCS"], var_name="sistema", value_name="pml")
tarifa["clv_nodo"] = tarifa["sistema"] + "-PROM"
tarifa["mercado"] = "MDA"
tarifa["fecha"] = FECHA_PML
tarifa = tarifa.rename(columns={"HORA": "hora"})[["clv_nodo", "mercado", "fecha", "hora", "pml"]]
assert len(tarifa) == 72                       # 3 sistemas x 24 horas

# --- Telemetría limpia de 15 min (salida del notebook 00)
tele = pd.read_csv("telemetria_15min.csv")
tele = tele[["id_vivienda", "ts", "grid_import_kw", "grid_export_kw", "pv_kw", "heat_pump_kw", "ev_kw"]]
tele = tele.round({c: 4 for c in tele.columns if c.endswith("_kw")})

# --- Programación del optimizador (salida del notebook 04): hora local -> TIMESTAMPTZ
prog = pd.read_csv("programacion.csv")
inicio = pd.to_datetime(prog["fecha"]) + pd.to_timedelta(prog["hora"], unit="h")
prog["inicio_programado"] = [t.isoformat() for t in inicio.dt.tz_localize(ZONA)]
prog = prog.round({"energia_kwh": 4, "costo_estimado": 4})

print("tarifa:", tarifa.shape, "| telemetría:", tele.shape, "| programación:", prog.shape)
print(tarifa.head(3))

tarifa: (72, 5) | telemetría: (48576, 7) | programación: (572, 5)
   clv_nodo mercado       fecha  hora     pml
0  SIN-PROM     MDA  2026-10-22     1  459.23
1  SIN-PROM     MDA  2026-10-22     2  384.77
2  SIN-PROM     MDA  2026-10-22     3  363.34


## 2. Conexión y creación del esquema (DDL, hypertable, compresión y retención)

In [9]:
conn = psycopg2.connect(**DB)
conn.autocommit = True
cur = conn.cursor()

def consultar(sql):
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

cur.execute(open("esquema_powershift.sql", encoding="utf8").read())
print(consultar("SELECT extname, extversion FROM pg_extension WHERE extname = 'timescaledb'"))
print(consultar("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY 1"))

       extname extversion
0  timescaledb     2.31.0
           table_name
0         dispositivo
1                nodo
2        programacion
3          tarifa_pml
4  telemetria_consumo
5        v_perfil_pml
6            vivienda


## 3. Carga de los datos

In [10]:
def copiar(df, tabla, columnas):
    buf = io.StringIO()
    df[columnas].to_csv(buf, index=False, header=False)    # vacío = NULL en formato CSV
    buf.seek(0)
    cur.copy_expert(f"COPY {tabla} ({', '.join(columnas)}) FROM STDIN WITH (FORMAT csv)", buf)

# Catálogos
cur.executemany("INSERT INTO nodo (clv_nodo, sistema, descripcion) VALUES (%s, %s, %s)", [
    ("SIN-PROM", "SIN", "PML promedio del Sistema Interconectado Nacional"),
    ("BCA-PROM", "BCA", "PML promedio del sistema Baja California"),
    ("BCS-PROM", "BCS", "PML promedio del sistema Baja California Sur"),
])
cur.execute("INSERT INTO vivienda (id_vivienda, clv_nodo, zona_horaria, fuente) VALUES (%s, %s, %s, %s)",
            (VIVIENDA, "SIN-PROM", ZONA, "OPSD Household Data, versión 2020-04-15"))
cur.executemany("INSERT INTO dispositivo (id_vivienda, tipo_carga, columna_origen, desplazable) VALUES (%s, %s, %s, %s)", [
    (VIVIENDA, "HEAT_PUMP", f"{VIVIENDA}_heat_pump", False),   # confort: no se mueve
    (VIVIENDA, "EV",        f"{VIVIENDA}_ev",        True),    # carga desplazable
])

# Tarifas, telemetría y programación
copiar(tarifa, "tarifa_pml", ["clv_nodo", "mercado", "fecha", "hora", "pml"])
copiar(tele, "telemetria_consumo",
       ["id_vivienda", "ts", "grid_import_kw", "grid_export_kw", "pv_kw", "heat_pump_kw", "ev_kw"])
cur.execute("SELECT id_dispositivo FROM dispositivo WHERE id_vivienda = %s AND tipo_carga = 'EV'", (VIVIENDA,))
prog["id_dispositivo"] = cur.fetchone()[0]
copiar(prog, "programacion", ["id_dispositivo", "inicio_programado", "energia_kwh", "costo_estimado"])

for t in ["nodo", "vivienda", "tarifa_pml", "dispositivo", "telemetria_consumo", "programacion"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"{t:20s} {cur.fetchone()[0]:>8,d} filas")

nodo                        3 filas
vivienda                    1 filas
tarifa_pml                 72 filas
dispositivo                 2 filas
telemetria_consumo     48,576 filas
programacion              572 filas


## 4. TimescaleDB: hypertable, chunks, políticas y compresión
`create_hypertable` ya se ejecutó en el esquema. Aquí se verifica y se comprime manualmente (la política automática corre en segundo plano) para poder medir el ahorro de espacio.

In [11]:
print(consultar("SELECT hypertable_name, num_chunks, compression_enabled FROM timescaledb_information.hypertables"))
print(consultar("SELECT proc_name, hypertable_name, schedule_interval FROM timescaledb_information.jobs "
                "WHERE hypertable_name = 'telemetria_consumo' ORDER BY proc_name"))

antes = consultar("SELECT pg_size_pretty(hypertable_size('telemetria_consumo')) AS tamano_antes")
cur.execute("SELECT compress_chunk(c, if_not_compressed => true) FROM show_chunks('telemetria_consumo') c")
despues = consultar("SELECT pg_size_pretty(hypertable_size('telemetria_consumo')) AS tamano_despues")
print(antes.join(despues))

try:
    print(consultar("SELECT * FROM hypertable_compression_stats('telemetria_consumo')").T)
except Exception as e:                      # nombre distinto en versiones recientes de TimescaleDB
    print("hypertable_compression_stats no disponible:", e)
    print(consultar("SELECT * FROM hypertable_columnstore_stats('telemetria_consumo')").T)

      hypertable_name  num_chunks  compression_enabled
0  telemetria_consumo          18                 True
            proc_name     hypertable_name schedule_interval
0  policy_compression  telemetria_consumo   0 days 12:00:00
1    policy_retention  telemetria_consumo   1 days 00:00:00
  tamano_antes tamano_despues
0        11 MB        2224 kB
                                       0
total_chunks                          18
number_compressed_chunks              18
before_compression_table_bytes   6291456
before_compression_index_bytes   5013504
before_compression_toast_bytes         0
before_compression_total_bytes  11304960
after_compression_table_bytes     294912
after_compression_index_bytes     294912
after_compression_toast_bytes    1376256
after_compression_total_bytes    1966080
node_name                           None


## 5. Join temporal telemetría ↔ tarifas
No hay llave foránea entre `telemetria_consumo` y `tarifa_pml`: el cruce se hace en la consulta, por **hora** (hora local de la vivienda + 1 = hora CENACE) y por **nodo** (que sale de `vivienda`).

In [12]:
Q1 = """
SELECT t.id_vivienda,
       (t.ts AT TIME ZONE v.zona_horaria)::date AS fecha_local,
       ROUND(SUM((t.heat_pump_kw + t.ev_kw) * 0.25 * p.pml / 1000.0), 4) AS costo_sin_orquestador_mxn
FROM telemetria_consumo t
JOIN vivienda     v ON v.id_vivienda = t.id_vivienda
JOIN v_perfil_pml p ON p.clv_nodo = v.clv_nodo AND p.mercado = 'MDA'
                   AND p.hora = EXTRACT(HOUR FROM (t.ts AT TIME ZONE v.zona_horaria))::int + 1
GROUP BY t.id_vivienda, fecha_local
ORDER BY fecha_local
LIMIT 10
"""
print(consultar(Q1))

          id_vivienda fecha_local costo_sin_orquestador_mxn
0  DE_KN_residential4  2015-10-14                    9.9758
1  DE_KN_residential4  2015-10-15                    6.2090
2  DE_KN_residential4  2015-10-16                   13.0184
3  DE_KN_residential4  2015-10-17                   10.5341
4  DE_KN_residential4  2015-10-18                   17.4774
5  DE_KN_residential4  2015-10-19                    5.7227
6  DE_KN_residential4  2015-10-20                    5.6863
7  DE_KN_residential4  2015-10-21                    3.4372
8  DE_KN_residential4  2015-10-22                    8.9860
9  DE_KN_residential4  2015-10-23                    5.0040


## 6. Ahorro simulado y relación N:M a través de `programacion`

In [13]:
Q2 = """
WITH tele AS (
    SELECT t.id_vivienda, (t.ts AT TIME ZONE v.zona_horaria)::date AS fecha_local,
           SUM(t.heat_pump_kw * 0.25 * p.pml / 1000.0) AS costo_hp,
           SUM(t.ev_kw        * 0.25 * p.pml / 1000.0) AS costo_ev_sin
    FROM telemetria_consumo t
    JOIN vivienda     v ON v.id_vivienda = t.id_vivienda
    JOIN v_perfil_pml p ON p.clv_nodo = v.clv_nodo AND p.mercado = 'MDA'
                       AND p.hora = EXTRACT(HOUR FROM (t.ts AT TIME ZONE v.zona_horaria))::int + 1
    GROUP BY t.id_vivienda, fecha_local
), prog AS (
    SELECT d.id_vivienda, (pr.inicio_programado AT TIME ZONE v.zona_horaria)::date AS fecha_local,
           SUM(pr.costo_estimado) AS costo_ev_con
    FROM programacion pr
    JOIN dispositivo d ON d.id_dispositivo = pr.id_dispositivo
    JOIN vivienda    v ON v.id_vivienda    = d.id_vivienda
    WHERE d.tipo_carga = 'EV'
    GROUP BY d.id_vivienda, fecha_local
)
SELECT tele.fecha_local,
       ROUND(tele.costo_hp + tele.costo_ev_sin, 4) AS costo_sin,
       ROUND(tele.costo_hp + prog.costo_ev_con, 4) AS costo_con,
       ROUND(tele.costo_ev_sin - prog.costo_ev_con, 4) AS ahorro
FROM tele JOIN prog USING (id_vivienda, fecha_local)
ORDER BY tele.fecha_local
"""
Q3 = """
SELECT d.tipo_carga,
       EXTRACT(HOUR FROM (pr.inicio_programado AT TIME ZONE v.zona_horaria))::int AS hora_local,
       COUNT(*) AS n_programaciones, ROUND(SUM(pr.energia_kwh), 2) AS kwh, ROUND(AVG(p.pml), 2) AS pml_mxn_mwh
FROM programacion pr
JOIN dispositivo  d ON d.id_dispositivo = pr.id_dispositivo
JOIN vivienda     v ON v.id_vivienda    = d.id_vivienda
JOIN v_perfil_pml p ON p.clv_nodo = v.clv_nodo AND p.mercado = 'MDA'
                   AND p.hora = EXTRACT(HOUR FROM (pr.inicio_programado AT TIME ZONE v.zona_horaria))::int + 1
GROUP BY d.tipo_carga, hora_local ORDER BY d.tipo_carga, hora_local
"""
ahorro = consultar(Q2)
print("Días con carga de EV en la consulta:", len(ahorro))
print(ahorro.head())
print(consultar(Q3))

Días con carga de EV en la consulta: 268
  fecha_local costo_sin costo_con  ahorro
0  2015-10-14    9.9758    6.2626  3.7131
1  2015-10-16   13.0184    9.4159  3.6025
2  2015-10-17   10.5341    8.7548  1.7793
3  2015-10-18   17.4774   12.4684  5.0090
4  2015-10-20    5.6863    3.9877  1.6986
  tipo_carga  hora_local  n_programaciones     kwh pml_mxn_mwh
0         EV           2                82  160.71      363.34
1         EV           3               200  445.24      357.08
2         EV           4               268  780.13      355.16
3         EV           5                20   22.59      365.88
4         EV           7                 2    2.54      371.47


## 7. Validación cruzada: base de datos vs. pandas
Los costos que devuelve SQL deben coincidir con los que calculó el notebook 04 con pandas (`costos_diarios.csv`). La tolerancia es 0.01 MXN porque las columnas están en `NUMERIC(8,4)` y se redondea al guardar.

In [14]:
cd = pd.read_csv("costos_diarios.csv")
cd["fecha_local"] = pd.to_datetime(cd["fecha"]).dt.date
cd = cd.rename(columns={"costo_sin": "costo_sin_pd", "costo_con_noche": "costo_con_pd"})
m = ahorro.astype({"costo_sin": float, "costo_con": float, "ahorro": float}).merge(
        cd[["fecha_local", "costo_sin_pd", "costo_con_pd"]], on="fecha_local")
print("Días comparados:", len(m))
print("Máx. diferencia costo_sin (MXN):", (m["costo_sin"] - m["costo_sin_pd"]).abs().max())
print("Máx. diferencia costo_con (MXN):", (m["costo_con"] - m["costo_con_pd"]).abs().max())
assert len(m) == len(cd)
assert (m["costo_sin"] - m["costo_sin_pd"]).abs().max() < 0.01
assert (m["costo_con"] - m["costo_con_pd"]).abs().max() < 0.01
print(f"Ahorro medio diario: {m['ahorro'].mean():.4f} MXN | costo medio sin: {m['costo_sin'].mean():.3f} | con: {m['costo_con'].mean():.3f}")
print("OK: la base de datos reproduce los resultados del notebook 04.")

Días comparados: 268
Máx. diferencia costo_sin (MXN): 4.9679999740703806e-05
Máx. diferencia costo_con (MXN): 0.00011314999975198248
Ahorro medio diario: 2.2285 MXN | costo medio sin: 11.059 | con: 8.830
OK: la base de datos reproduce los resultados del notebook 04.


In [15]:
cur.close()
conn.close()